# GwenLand glcuda Wave 132 - exact Q8 decode glue

Production A-B-B-A decode fusion gate with graph replay and a 50-token glproc oracle.


In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import statistics
import subprocess
import traceback
import urllib.request
import zipfile

BUILD = "wave131-fused-swiglu-production-ceiling-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "3e2622b7137f014f11b6892000b5c7be74b67c5f"
SOURCE_REV = "76e16c2a202933fc72df2a6890699ce938e8b176"
PATCH_SHA256 = "cdf85121d0f788d97fe942224b9ae3acbe0f7479bbde85e42ee9f72fb12fbe35"
PATCH_GZIP_B64 = """H4sIAHBPt2oC/+09aXPbRpbf/SvamiqHjECIAG8qzoxjKy5vYiu27GSqFBfUBJoUIhCgAVDH2KraH7G/cH/JvtcH0DhJZZLa2UPlMkX0/frdB+T5yyXp9VZ+SujRKnC3Hj1it3S9CVhydEOvmWUPnE9TJ4ySNIqZGSdksV+/R37osVtiecOJN50MpsPFxJ7gv4Hbd0eLhb20+v2Z685mQ8uifdNk48VkOpot2GDErKVlj9nAouPpwGPupE8n45E9XUyt2YTAuPFw+KjX6+2750eHh4d77/tvfyO98cQYk0P4f0Lg6zIka+qHnS7pfUvesWQbpN90ugb5Lrr9xrsLSZJ68zmL4yiez0/w49tvyedHRP0sWOherml85XhsyWJnuQydmCW+t6XBnJxFa9ZJ4y3rGnVD8r3JrksaJA19ceIVTZmz3ThJSt0r5pUGHVYHhdbYWW4T5jnJjb8KtnXL3HePxS+rzdb0Qz/tdP+qPQki6jnryGNB5wn/wFZvJ2YNrMLCziaOvK2b+lHYgmatgx6F7IYs/YAR3IZCFIGI/T1/TJOOByPXGs+Wi5nlTeiAepPJaNBfwsPJwhrbs8FwZrPxjGPgkceuj8JtELThWPumEeH6Rp8cWoY1GQHCPTo8OnpMfoGRBIYSP+zBAJclCckHkmdH35FlFMtu9uyrhPA1yNmN//LHD+SKxSELTD6XmPC7KL0kNF4n0MY2JL1kJGYpIDYMkrMMCA3ll+mUbGh6mRAW0kXAPJO8hwFRGNyJ2Vy6TWhA4ESpe0n8hNxcMpgy1ve4iRncRgDLbAIKB+BrSswkiKlH242Y7eXJ69dkE2wTtf1PWxqm/j9gPljgUj8lgNZNe8kl3TB54oBuAZvhqI8O4TuBK0BymbNwBWdz0pj66Xz++WVwwh8Y5FUIZPgq3GzT++NsCNwa74S/PI/Cpb8yiPgmhuVd8TLm85f8U7Qd49JuFCYp+end6euf3jsf3rx6PydPkjQmT8nBa0aTbczg+AAnj6UsXgMVJanvkuQuSdma3+x6kwJUAWSATXcmObkFoPkhuYxuSBpdMcACGtMgYAFgRMpWAJo1TWP/lqyBIfmbwHcphzrsMiEAnxhAs2brKL4zyHsWJoAszwEwicEveenfZqAjKwY0n8aw6sGxOsnbD8/evT95fzaHCeEi4BxWP2v85dm71x9+OnN+OnnnwK95n1HW5eTvP508f3/ywpEgeX/6w8kbbTZ7OORwA+7q4110fC8BkJ1vB/ZHzmnzawJuyjlX9Qn+cOA4fDT8Z6aRc83cjs7t1vTWAdbg8J7QzdLaAPobFtN0ixy2b/b1pmjjXM3JsPxsgx1nI+0pIDijqbMBWgnSO1jA1JcwzXzj8zkIAQoX1umKDvePDu8lGNbM82nYWcPxbmGXPzP3m+V4+C0HBvyiznybmHCXqbO463yhBll8IdQE1Eh9GjjuetNZdM1teBPTTacLfFsOOYdRAUMZdkTsj/ma8TZ0gCd0RD9BMnALOubLk2jXI5+4gEe+B2Q8J4soCuTTKKZuAI+gIzzRZSacYZfQ1HZhJgzO2CinOtnqKG34uIClJALgPVUz+Aj2ToZd3aynv8SOpiA6iRbk8dMGnCVfvvDuGaKZNHESoDcG4IRR5+LIH3WkBM66jUMCJ+sAlwZCfdzJG/HnQAwiaz+BVvdyjqz56ed7UtgUPlC/zf96D8xvw9yUeU/PP99/PDCKU8KpMqCQz+Qg+3JAYCTIc3yoWD48Kw2vQKSmPQNBuY0fRnvY1dAf2FWU4+K9+Di96ogFuYhw1klXI4SHa1sZAiD10HiVABaInuH1fI4POl0zufI3HauroQtXV6ArdtB2HLLbjED5g+jKieLOAYi8FaD2DqFOXp++OPmRnNMviy/wFBWSjwcFLOXqiVhVrqU1xuyaxQm2c8RgyeMO9kek8xiAC4HBdbSDxUFXHyjXcqLQ3We07C7mKMyCkuipLspMweA60wLsANKg/kFPnVvM5yiwHZeL0I4uT3XySFimmw7tesV0l7J8uIey3Ny1WVduHrNDVT5UqvJhRVU+bFWVM4gCXQE0sR+sCa2OewlAfyJuRGdeyIMEN29mWg9iRkr9iEJQl9yUeLG/TOcEuA8s8Pm+wmnU+qXn9VvZkysUMcuVmJVrWfM5yPCMUtwKfN2d8BVcyhEyQkyQC4gnBQmRd4euX+cDtQNkvFB7tvTjpJZzCLVRTbliIeocoICFkeDvB4VjJJfb1Ituwk5Omqjr59wd9MJzjncGQXwtiB7s6WAPsGVKeprejSOGlP5PAOcMghAw8kUMudtsZxr3zq4LkLHAdrQFBDNR7B3AWLca7r52IRydsoBxvVRQRXHvZtaqw5u3wAkcMDXCzpf0C8lkTLmXvJnMzlLWSr7qNqTX1A/Q/jkobG4TA/YGYYWMzqVk6Cm2Tz5//vWgIFR/PZh/vjd+PbgERuXk4MHHc3OGLXDS3Q3pRraMsUXiJ1+BL3C/N8nmK5UaMjAAoqe0pkPOhGjC9dOv0eIGFRpUzJ2DqwrDcRGDgSmvOJ7nU/FHSRmHG+6ieB98qLiNkK4ZwOjXg8/3vx4A7NQGNUC7YGZlF7W4S1kCMoh66smaukkDlLlsw7VMXKaxsQEoeQe+g8bWfEtS00dE7ncb++OG23p2qwSuBFaBE5XkCahwDQw8RCmDepXOpNT//NHHqsLjYX+tS/bBOVyBL4MmA52B9yhd6XM+Sabqyk3c6wI2vI6klVwaDrqUpiTTg8IwlEeZq+QpGmZKy6Eb6vrpXUcZy0ADdllHyrn23kORAD5twabje0ZGnvUqMfrOJkp8PJCB9oOTG0U4jgOKC1rHT0HQdUGv2K655AGqLdHRXuy6sEYt31ZTQU8B48KQ8qKcSnM7pdio7qNmSGbFlEYc/x7ewF11gjnkCKKzCHkViv4VyNV3OKHeu5bd1zL0383MOfvNtlrTKHdc05LhS7UJzlG3ToPgaBUeu+RD68CKbCjzKKl47MSsrNHcbJPLjmZrHu+LaQrRds5QpyLlDE5M4ggPD9pmwtWTze4GUcg6BWsu23xlVH6s0rAadNfQfLsGK+ZuDzzPkFZQRuJswBATSM4fl45Tlp3FbRdbs6FFlM8HFZ8nG7ATt5v9iaOWKBTvLDjsJODLmF06m9ZSPpjW1GCBVdG+efa9p2jZRhnNWjsXiEw3WisSX4h59M7UxXSS2D0SsYbkCCyuQuim2iaDMQtr6o1Hi763mPaX4wlbTu0RhlssNpu5E3c0YMPFeELHpjmaLuloTKllucvlzB25zFpOWb+/HNh0vGB9thxaY29k10YFa9YvRGlq2jEYY9t9Y0YO8cMaYABws12AJhVvwTD+gXc/Y6mK8CF5jWZzcrpBrPvmF/71deRtA/atDJ8dHR1hcw8ksgqt3Pge661i3yNvrDFc3MpPQED3kL8wjKfkJJ6vMp0WVplOs1V6datY9kzER3oYtCFASq5PA/8fgjpIiKoPSUC5Rn8iWdxhcCafSfNmgR4A1irJ5h30TYEYD1rRzANaa3onFyY+6jf5bBgfqq6Meo6fysgP2fghqmE8AKRBBxYvgAe+Z/DJ5ldbmKuQVF1ESgtGYesyCoLoxg9XMjSVz9Yco0qgO5j5GQTyIBUg0AaoJ43WGPyJGo+O4b1LCriWJIgMKs7Gw3eo2YWMPP/w4hn32LDbVN5I1UP1jK/0HXfPZ8j4SxaqExZvchltAw+dPtcCIrE9GpMO/NeLo5uuAIgfAopSj0RLjinZZOMhdsIdmeQd74B+AJoS9MWQJShD5OLlj7hZ5x3MeHGMzr98+PlFRlLzOa7ryIDjxUdTUOPAMqZAjYORYVm7qBFnVJDON1PYxMt3r17YL1q3gYRpe/pGhD3EH6tgR6+IVJZET+r9BngVpj33EqN3cBtwnccK1IIyMP4WccIxy7i5Yxoxqhhj9deg0F0rWsDrR0co3H4xLqPhv434j8FMvGuxnlodh8nYMSJaAnyK9wJzBUNkXj6XwPEel8TEY+ivBAjRzSVGgsPsAIihCRodCSAvI2+nvZCK7fKZxMDyhklxw/ZgTtB3T4Drk8SN0bFNbmKwaRKxGO4QZRagBaPrDK853qKZnOQz8gvA/m+nBA7uXwtz0I02dxKCuhdZJx9ExsGgj5khg8HEsIYyNyRmSRRcsxqHdad6JeTJE82jHAEPBqRimnHOwms/jsI1gE4hYPcRKJNEhOvkWpVoWM0wCcya5TJGiR1EfBF/U/rvw/eXhxMzWOjX2qlBSnXzMfu0ZcBZ5HYrmykDrzxMrE180FTr2cG/vXov8Hy9YJ4HnPin93/nFCj3SlAQ3hE4D/y/iUCDNskpMNy1MxkdFlldyqPoPUwvwBAOCBdMfEA+B9OmUUQ6ism8OXVev3721LpAik0wdGagGBM4NBwZE8ChcV+kF9VvnaucmJRAnjyHDy3hp4pnDRcrx+iBtGwZTG4Q0TNYsKemPmPBcj7nTnTupKhiQtLBTRltmzDImyiU/2s+1z98brWCNLpIfk8/ouTBOxeMjCRbYBVCSGerCk6cLc2VDJHwkl5SdNwkqbitEc8FG0zsXbf10Hsp8prdfWtCVo2DMqiXGcU/iyoFH4+kgaccUbPYCzzqAI1lKWKqd4IOIX7FGHeJTKCwNf0tig1SfOaHUQy2BwJ/aM0Q+EOU/f126PPEgcwAPjgXKv5HYvdeyPQZrm4XRIPkXwddbaf3xU0X2I8eUr6msRMlnQNJ8N9/ODtx3k4dUBBPDrqmnzgJxga7pYhGkS+ik62eYRadC/omSm6Rxg29OHl++qJ2S0Wne+G0OTrC1vRVi3AGNlyDuI/KvhOSiw0H/SudL1+a9wsbfXN69v70XWGvCg9GQ8OyAREm/LMVETJfTmEryCoe6X7Cnn7uMpVgvImt11zMKpuseJEVusoEUu3YMvj2hG11XJUHVPo0ghhMTY6nL5yzX14BVhTxtAAQaVEJD24FOqUgX2tnDTClK9nJQZyz15OR88uzn09g9gI70akzv7MqAoq5zBVLYU8hV5gxFOvwC1qvKV7BgsdIKhqVoxlvPPjH8XA6RVtk1MePnezIIU+/xVwzHe1qeFUOeeGwE5yrp2LxwmHHtXRHmAHKD5fjiXpSgx6qqSqdVMvqE3VWcSTcbPwBt3Mybx/YZOp3MHcDuOppFhvjsMvGSbQvfh/Y5faMLLJpS7DPHOtpCtuNbhIHpLObnwV2PHG4YZQUusKNDisP4LyrT9Wn9Dp3H/Yama1Gi0Yd8dUpKwbJIGpIm9HgFrVBFPxAKeeQMzJWoX4b2EaVfRgVqoIJc7DwBRVADCKOl4KyKjw9GRAaHl4jIWRN5Yj23thYlGD/j6N/MI42KwQlpeJfFmdzSfAnYi+pU23q9ELvLqRg2vdkorTSC7OZapXD06uOzu25SJhN0Fk8spRHoF0mCHexUX0IkqX6EORe6ebbPHwiQarRRVANbrXc0B6dM5usFKjrlg7C0aj4SGJY8WGLllvC8OK4en+NAEbeVt5VbQJgeXhNp/I8Od1wdJj0ubk4AZrYpagmYBCbVUWQu3rKnlrluOO+c+W7Fp475XpDe0IQkbB00XORzUMDdIPdYQmBn971UqGQifKJt9PM4edtY3R0CzeedPlJRx16XZdh6SaUB6jzBA9T8d1kpyyOyhNGSP057YGIETS4/NBUfzsVBryi28zLKJ14cruamdK41eKFaMVgqI52TjGHAoAyn79jAb1Fbxe/Z7BMJ3DPo36TJpgf7t3rszdRvJaRhCzYQjCNBwTrekNTH7ZW8khm0QWD32c+HQ3RqQRdaMBrgzguKEZOqAfzs2UUC78tusJpuIJFYRfQS9Ud5e7r91iU4icFD/W86NBV5SHcbaY5yhIMndyJUhLNk/1CjMUrpEESEeFp9hM5lHtsFUKrgh68UI7C6IOB7Wh+ZhmiOuYxIDDjeLhBFq7EbM0xnu8Ts7xS35UY8JdziuDuuIG/2dzN52kUOWsa3jk0Xm3RfZl0PxZwJV4njgI6oirKqE6OHRxvBJFbM3uAt2/NRiNg/juuX+E2XocPvJnnfWYBM2kBuchNE1kh5McYG0PGUwhzyX1WGHEBsw1y6XseCx3PX4siDEAS7UuYRlf8V50EegUSkJInNw7RqqzaR7izTr6YWkcs0S1xgMqcRQaPZit2K6/SSIPl0Q/eXtVrqO4pjMLe0kfqUswloHd4OYjQXHXCq/Rj8v33b4p0h5lWEkFGAxQD1mwKasFot1pgco7TqYgEYc8LX58URAapguO4RnSciYgh55Iq7pkVCapQJ6e2BUtvGNBkcgckFUchmrz5PFqsKXOeEkwn42wqEXWBEovRa0oucbEFTig944u7YphTJuZpkYQ88TxjXUAHMuaJ4U+B4rCsZDN/OfciV94ukLEuoXaUDUkiKcRIuhWBVR5WSh3WbquWWOTs9VdVlX4IQX5F9kTLDBDckl+XFBg8QBvDJUauu93Q0L0DKcHiuyJ/wB3ZE35y1AW3oGQ7QsBrsk86fwVCfluRgqBUY8UIrxcx13TT+SJu+ItyqeReEyEPB/0Bd9UO+lPbsLnmAz0JKhqJPjtNEhanjzuPW6JmhZzRvCajW0Hzv5zj/PL65ckrdTlKTUXLBNa+ZgnmhmvlOk5WkaejQWWjFUyS2+RBCN3Pq0Y2DhQHaxz3eMdArejk96xbBGgxd7ceoHbJO83NBJBRApC6L8BRdLwDluXwoJbu2w6S+oHiNA8Yp2UZt4KwdlhhtQodY3TRX2PuOmeGMEOAcW8OJRl5LmhQmUjIp5GrJ+Q///0/gPvxKi5RkZOxapzG3KS3otod1TLOMQVx5jOh4ojVLiFjHir2lLz86YPZmMcV+ItK/pZ4pt7mMJjZo9Fwslz0R65t9/vj2ciyxtZ0ZM9of0gny/5iMJyOTXM4Gixs1+pPFi71htSaTl2we5fT8XTpTa3pwpot+ouJTRvztuS6lXwt+Zy/pWHI39IwFCqYlhZSqjrLwPFdffBPr6MffJVk+UAo33VBmSmWBdmTsd4dxWU1Abr99sTFtiaFC2X+Yl+tMrGtgK0UA+T6kEzrEgbGhsUyxQNLEhZwFsA0tCUx4g3KwnZ1Sc4vigWA8TZUyTvWkOdLWMOBYeV2kt5dFw4y07BGlxHZYoVXFmjwebhSk+svvMQEZStQiMtV81qFR+Zi/UnqR7HSVQu3FuIsykp/ymV0SSWRjeVKKCnD60qfspicXG8+FzeCERK8H9DTMDvPT1UURJXsFfL95bq7qrVVqog2UuWVVtjoCS9+JBeY2HZBUlkhx6sjlWdDqZa8xg/Ig/cQxjIo4sDqtKQN6P3d6ZnoYpJnqqKL0CUgAUdehbh59eDFx2OBbVswB9caRxW6KzJyuWJCthuRdmSQBbDaJeBa2mMhbJZby/JFICKr4DIKPKEuWSKPyB5pWQVqI5z2m2gkU9FEha3ZVjBr7DNM8xbv073OoVwKSdeOq0QwNFdtOVPABxR/kiUFaI3ImUKesBmnQOgBf51H4F+p13IAJpwn/tr7yJvmBIuw+D0C+Xo3IIcbRZ9INa1Iv+yxFIDeYAqyzxpPxpTa0769WIzHlu3RMci5KVvQmdv37PFoYJp925561LWG0M9i46m3HI+sRZ8yai09Nh6N7eXEnS37jQIwX7oiA/Mm4XLk2RGH+DnUmOxm+5pXt5eSNj45G/5WkpvE/HR1bXrwrQT+K9mDph2tkwEjwX4upwxct3Umh+TqWo5qz4O4Mhs8i+WRWFG1AfICGi/gOsyBGnVWLytrREoLfS5nT6DDiPAQM6i5z968Of3w5vnJi7kIoqPsmM9PRTl96UmxGln9VGbilXy8NBY5bU2BCWupUSpE4MRJZEUpSDheuCEV9J54Kc4n4UPjtRzcI+aIA0JXVFrv7w+qa1QKWYrFgETLP8Jys8AQ3hBeX8YpXThHzNoCs8JKQLwcybMIyyKIQJ2AZ8UA7JWJnjhktDIf7FauafLIGDZIHLsNDcJdOi4fwjZJbSkaICUvlkwADTDVQLiK8ZuDpsoTMfnNJ/OmW4UPWPNNA64eOuAaBjTsDzX037U/wKqH7a9lQM3+qiSECSo6QGtwGm76e5B0MqedXfvRNhFX+FUiPGixVK8z6kXVgtswtbOpsTnuIBYYuEBI2BqYuW5fvZ1y13rCU39r51vckbdHPxz9LJxXQqbzLH092sDf1cRTyIVxpanaVXQtO46VTG2gao7Y9U23Dc/rmJ6Z0itWW5nHcauOaBoWDRsakKdPnU9J22DRJwFOx1r7Iak2gEPRb0O7VfO8QurNRXyAxEhn3POhQNcFSmsBad004rZBv8z5Uh4/RyFX3lGpMPBPYXAFFi3R+9PR1dE1x35OD2fPXp/oSK4FmS5uwwtuc4pCGLpmlcmU2zyJRAQjTKc8Wx7pYw1KuKz3AF2X4ssTwDhkmF35s3wHmVBR7IFQUcaWru/WaChF3AWeJ5YnT58iIJD5ZMytqeW62GJUJz8QAOIOGr5JecjcvJBZAYqqwRAMopuDR2XJ2Su/fqmNdfceyrp7D2TdxQGfK4y8IGcAXo877ay9W3cxV6bG6hT+ApoWGUUNT1AUUqKOYjCdra+dG/TUypmvcCaDZHAUa3EVVuXkTYcDwx6Qw+lkakynO5GLb6Fu1foVI34UTpf4uqrscGBt/Ma/4Xy9Fg6h981hUKsZSZfOEq6oB8oW2CuepiOVR6AhhHSWk2/uBTkE4+iG+avLlAjbNJGULkm1MpmMrK/Eu2VAFJ/3VSCxy2kbJhTPxUOD2F/LZnNf7Q257E7eVknxXIkXJwAYpdVZtlhUR9ihskL0njh/HgUs02zb1ecBN9zqSr5prnTYxA+2znobyClW/LUIOEZfFct4MeL614esjvVMasJ/GuF263RNjEVBAfhLg6qHKlSmmPWASjb8dRlKMKKaJ7Q1LiMy2VM7V15SiVwZqxxFdoB8myl5KV18arcivPl2atTOJkVWg36HrJD7uBaYqwDmi/Rz/TfqeFxDKVx2q2pXpKj/vZrdPqKhnlr3VBAfTEJ/AKdrOOHvP53u+Xg4mSO7aadx7IHO8d+YrM5GNUHVU4IyqMgj86Af8wyu2vmQLCPMtrjxE9bjZhaSppSAXBphmMyPWXBXS4+K7xaI8nfTI+ey9U2Y2Pen00+NsPjD6KJNjuxNHg8Vcw9D7v23uBvH6/H3Qk57kWWMBVjrjt79QMgIdpsK6s1li/Zyghr0FdqWknOHuQREys8EFi/SF1l0IWyz3r2hMhPUC6LRLmWY3aHit5jJhq81wRBuyOmGLLbLJQCv5p089c7Rqmj587jj/aN68xRu6wao1Qmilc/TQnZqjtzDKFSK3Rx1t26lTZdZLWIz9fi2HwvVZm3goP+6PqJ6AP8fVSUeih7/oz1ODyKt/fj5g+GnRRoEKB8CtzLAMPNYZM5iBidfF6wH5m65tpKkPJOT0Wv5dwiyaeRLI8qsXrJ59IkVzAMeNGIo6wrpCDXe4V03V8cviykQPIVC5VDY8GnN9vKeyTgEylVkNvyvNjR1Ktt5cH/Ke9axyde5F4AL972nKXrnxOAaPMqrOOVAHmkrJLzWkwUseWU2pkKLTStVStuCQdTTbuO85aOIXMfamm4NWbKMbpeGaFWiW0YosijrcuWYv22Y3Fxi4taGsbh2sufvn4l8Y4G23JXrpyZ5JV4mgTOgFiCSZfm7XJK84KM0V42erTJlkoAuEoQHxkuOOHMmKJLguMeN83E7gNfPZ5oSaClAvDyjDROlZRIv87mmgouQhsAOAijFi1oBFZmiFiHL0+QE9g8WRz2l//AXtYgMQ7Mel3jJzqdEFL4JadMVhclFLGtgtp3N0hF6KPyy3dTgSDPPV+NxdfiQi9fMUPH/tURHO69tczCX7yjJk7GiDbrfT9+c1E604DmD2V/3yByE7uU2vPoq4dV3Xe5UFDlaFn8hiTWyp8ZgsjeDua8+vu8eVx9uLmnCHndSZ7U1yNl75+UHo6z6aWKzRPxNgN73QitRfusPiPJXI/7WAyL+D4n+12YCYK1lXR6APeuJ168JZtjLE9l4RkBuqKlKU7/wDesiWt43Wv5RjmBRU9Hevw2A922N0vXwXU++Dpon1gLmY00qslLyxrKnglQS4R2s5TRl0w3f0k4uabBUvu0FW6FRSFOt/eYyAnaIK7VOKNLILgQ4Lo41OaC44wL/8JIo6hEv/2qdT7riq6+oUn/7I6sGUHzYbJ6Pv0Wdw+ppk1jL0vzF/J2D1ZbGmFtHF8A4DtpuhzPc7QY4nhMtlwm+KgazwpDxyQfAe9sxQ9YRpH42JGkW3t3dWJmdIrsG8ZeiAGcQVY5+AOkHetwqFJW9jfNdmXu8I2IH0bYYcOqH343ZbjkV++62oGrnJoekcFEPXEoM1292xwxCDu7sI5bY0U+K9N2d9pqtiF3tfVvdcYXi6N3das2ooulTlX2NC3LsLFUoNuNga5crWdQ4HI2FGjCbGbM/TwtgQcrVgLNXP7YqAleFQrg8H/CP0Ah0dV2WRhO62QR+nljPk3fXqJ1ihbTTN3ddXst+G0H/sDtv8XyrWTKP0i4+KRFDXPyMv9XGGg8HxnTvi8effgtq6bTTxgKE7723g5H0/hkmspOB6Myjba/o92zpEbYfGDnAfwEQXXuIcXUAAA=="""
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q8_0.gguf?download=true"
MODEL_BYTES = 675710816
MODEL_SHA256 = "ca59ca7f13d0e15a8cfa77bd17e65d24f6844b554a7b6c12e07a5f89ff76844e"
ROOT = Path("/tmp/wave132")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
MODEL = ROOT / "qwen2.5-0.5b-instruct-q8_0.gguf"
FINAL_ZIP = Path("/kaggle/working/glcuda-t4-wave132-decode-q8-glue-results.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)
RESULTS.mkdir(parents=True)

def run(cmd, *, cwd=None, env=None, timeout=14400, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({k: str(v) for k, v in env.items()})
    p = subprocess.run([str(x) for x in cmd], cwd=cwd, env=merged, text=True,
                       capture_output=True, timeout=timeout)
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    if p.stderr:
        print(p.stderr[-12000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p

def save(name, p):
    (RESULTS / name).write_text(
        f"RETURN_CODE {p.returncode}\n\nSTDOUT\n{p.stdout}\n\nSTDERR\n{p.stderr}",
        encoding="utf-8",
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = sha256_file(FINAL_ZIP)
    print("ARCHIVE", FINAL_ZIP, digest, flush=True)
    return digest

def percentile(values, q):
    values = sorted(values)
    x = (len(values) - 1) * q
    lo, hi = math.floor(x), math.ceil(x)
    return values[lo] if lo == hi else values[lo] * (hi - x) + values[hi] * (x - lo)

def bootstrap_ci(values, seed=118, draws=20000):
    rng = random.Random(seed)
    n = len(values)
    medians = [statistics.median(values[rng.randrange(n)] for _ in range(n))
               for _ in range(draws)]
    return [percentile(medians, 0.025), percentile(medians, 0.975)]

phase = "bootstrap"
try:
    embedded = gzip.decompress(base64.b64decode(PATCH_GZIP_B64))
    if hashlib.sha256(embedded).hexdigest() != PATCH_SHA256:
        raise RuntimeError("embedded patch hash mismatch")
    patch_path = RESULTS / "wave131.patch"
    patch_path.write_bytes(embedded)
    (RESULTS / "source.json").write_text(json.dumps({
        "build": BUILD, "base_rev": BASE_REV, "source_rev": SOURCE_REV,
        "patch_sha256": PATCH_SHA256, "patch_bytes": len(embedded),
    }, indent=2), encoding="utf-8")

    gpu = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
               "--format=csv,noheader,nounits"], timeout=60)
    save("nvidia-smi.log", gpu)
    fields = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"requires Tesla T4 sm_75, got {fields}")

    phase = "reconstruct"
    clone = run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=600)
    save("git-checkout.log", checkout)
    applied = run(["git", "apply", "--whitespace=error", patch_path], cwd=TREE)
    save("git-apply.log", applied)
    diff = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff)

    cargo_candidates = [shutil.which("cargo"), Path.home() / ".cargo/bin/cargo",
                        "/usr/local/cargo/bin/cargo", "/opt/conda/bin/cargo"]
    cargo = next((str(x) for x in cargo_candidates if x and Path(x).is_file()), None)
    cargo_env = {}
    bootstrapped = False
    if cargo is None:
        bootstrapped = True
        rustup_script = ROOT / "rustup-init.sh"
        urllib.request.urlretrieve("https://sh.rustup.rs", rustup_script)
        cargo_home = ROOT / "cargo-home"
        rustup_home = ROOT / "rustup-home"
        cargo_env = {"CARGO_HOME": cargo_home, "RUSTUP_HOME": rustup_home}
        install = run(["bash", rustup_script, "-y", "--profile", "minimal",
                       "--default-toolchain", "stable", "--no-modify-path"],
                      env=cargo_env, timeout=1800)
        save("rustup-install.log", install)
        cargo = str(cargo_home / "bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable after bootstrap: {cargo}")
    (RESULTS / "cargo-discovery.json").write_text(json.dumps({
        "selected": cargo, "bootstrapped": bootstrapped,
        "candidates": [str(x) for x in cargo_candidates if x],
    }, indent=2), encoding="utf-8")
    common = {**cargo_env, "CARGO_TARGET_DIR": TARGET, "CUDA_VISIBLE_DEVICES": "0"}
    candidate_switch_env = {
        **common,
        "GLCUDA_FUSE_Q8_GLUE": "1",
        "GLCUDA_DECODE_Q8_GLUE": "1",
    }

    phase = "host-tests"
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"], cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    if "74 passed" not in tests.stdout or "0 failed" not in tests.stdout:
        raise RuntimeError("unexpected host test summary")

    phase = "cuda-parity"
    parity = run([cargo, "test", "--release", "-p", "glcuda", "--test", "parity",
                  "--locked", "--", "--nocapture", "--test-threads=1"],
                 cwd=TREE, env=common, check=False)
    save("cargo-cuda-parity.log", parity)
    if parity.returncode or "0 failed" not in parity.stdout:
        raise RuntimeError("CUDA parity failed")

    phase = "graph-replay"
    graph = run([cargo, "test", "--release", "-p", "glcuda", "--test", "graph_replay",
                 "--locked", "--", "--nocapture", "--test-threads=1"],
                cwd=TREE, env=candidate_switch_env, check=False)
    save("cargo-graph-replay.log", graph)
    if graph.returncode or "4 passed" not in graph.stdout or "0 failed" not in graph.stdout:
        raise RuntimeError("Wave 132 graph replay gate failed")

    phase = "compiler-resource"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    resource = run([
        ptxas, "-v", "-arch=sm_75",
        TREE / "glcuda/src/kernels/glcuda_sm75_wave129.ptx",
        "-o", ROOT / "wave131.cubin",
    ], check=False)
    save("ptxas-wave129.log", resource)
    resource_log = resource.stdout + "\n" + resource.stderr
    expected_resource = [
        "Used 80 registers", "used 1 barriers", "16384 bytes smem",
        "0 bytes stack frame", "0 bytes spill stores", "0 bytes spill loads",
    ]
    if resource.returncode or any(item not in resource_log for item in expected_resource):
        raise RuntimeError(f"Wave 131 resource gate failed: {resource_log[-4000:]}")

    phase = "model"
    urllib.request.urlretrieve(MODEL_URL, MODEL)
    model_meta = {"bytes": MODEL.stat().st_size, "sha256": sha256_file(MODEL)}
    if model_meta != {"bytes": MODEL_BYTES, "sha256": MODEL_SHA256}:
        raise RuntimeError(f"model identity mismatch: {model_meta}")
    (RESULTS / "model.json").write_text(json.dumps(model_meta, indent=2), encoding="utf-8")

    phase = "build"
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave131_fused_swiglu_production", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)
    exe = TARGET / "release/examples/wave131_fused_swiglu_production"
    prod_env = {**common, "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
                "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_Q8_NOSTORE": "1", "GLCUDA_FFN_GATE_UP_STACKED": "1", "GLCUDA_DEFER_FFN_RESIDUAL": "1", "GLCUDA_NTILE128": "1",
                "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
                "GLCUDA_GEMM_N16_PREFETCH": "1", "GLCUDA_N16_FUSED_SWIGLU": "1", "GLCUDA_ATTN_MMA4": "1",
                "GLCUDA_ATTN_MMA4_REGQ": "1", "GLCUDA_ATTN_MMA4_AV": "1",
                "GLCUDA_DECODE_Q8_GLUE": "1"}

    phase = "production-ab"
    records = []
    summaries = []
    for invocation in ["a", "b"]:
        measured = run([exe, MODEL, invocation], cwd=TREE, env=prod_env, check=False)
        save(f"production-{invocation}.log", measured)
        if measured.returncode:
            raise RuntimeError(f"production {invocation} failed")
        for raw in re.findall(r"\[wave131-sample\]\s*(\{[^\n]+\})", measured.stdout):
            records.append(json.loads(raw))
        found = re.search(r"\[wave131-summary\]\s*(\{[^\n]+\})", measured.stdout)
        if not found:
            raise RuntimeError(f"production {invocation} summary missing")
        summaries.append(json.loads(found.group(1)))
    if len(records) != 80:
        raise RuntimeError(f"expected 80 production samples, saw {len(records)}")
    if {r["oracle_token"] for r in records} != {3323}:
        raise RuntimeError("oracle drift in production samples")

    retained = [r["prefill_ms"] for r in records if r["arm"] == "retained"]
    candidate = [r["prefill_ms"] for r in records if r["arm"] == "candidate"]
    if len(retained) != len(candidate) or len(retained) != 40:
        raise RuntimeError(f"bad arm sample counts: retained={len(retained)} candidate={len(candidate)}")
    retained_sorted = sorted(retained)
    candidate_sorted = sorted(candidate)
    retained_median = retained_sorted[len(retained_sorted) // 2]
    candidate_median = candidate_sorted[len(candidate_sorted) // 2]

    phase = "candidate-profile"
    prof_env = {**prod_env, "GLCUDA_TELEMETRY": "1"}
    prof = run([exe, MODEL, "profile"], cwd=TREE, env=prof_env, check=False)
    save("candidate-profile.log", prof)
    if prof.returncode or "[wave131-profile]" not in prof.stdout:
        raise RuntimeError("candidate production profile failed")
    profile = json.loads(re.search(r"\[wave131-profile\]\s*(\{[^\n]+\})", prof.stdout).group(1))
    stages = [json.loads(x) for x in re.findall(r"\[wave131-stage\]\s*(\{[^\n]+\})", prof.stdout)]
    expected_names = [
        "qkv",
        "attn_norm",
        "attn_kv_write",
        "attention",
        "attn_out_quant",
        "ffn_down",
        "ffn_gate_up",
        "attn_out",
        "lm_head",
        "ffn_residual_norm_quant",
        "ffn_silu_quant",
        "ffn_residual_add",
    ]
    if [s["name"] for s in stages] != expected_names or profile["oracle_token"] != 3323:
        raise RuntimeError(f"profile contract failed: {profile}, {stages}")
    stage_sum = sum(x["total_ms"] for x in stages)
    ranked = sorted(
        [
            {**s,
             "share_of_gpu_total": s["total_ms"] / profile["gpu_prefill_ms"],
             "share_of_stage_sum": s["total_ms"] / stage_sum if stage_sum else 0.0}
            for s in stages
        ],
        key=lambda s: -s["total_ms"],
    )
    by_name = {x["name"]: x for x in stages}
    phase = "glbench-ceiling"
    glbench_build = run([cargo, "build", "--release", "-p", "glbench", "--locked"],
                        cwd=TREE, env=common)
    save("glbench-build.log", glbench_build)
    glbench = TARGET / "release/glbench"
    ceiling_prompt = ("Measure this deterministic systems prompt carefully. Explain how "
                      "token-parallel integer matrix multiplication uses shared memory, "
                      "Tensor Cores, and fixed launch geometry. " * 8)
    base_env = dict(prod_env)
    base_env.pop("GLCUDA_DECODE_Q8_GLUE", None)
    ceiling_runs = []
    for index, arm in enumerate(["retained", "candidate", "candidate", "retained"]):
        out = RESULTS / f"glbench-{index + 1}-{arm}.json"
        env = prod_env if arm == "candidate" else base_env
        measured = run([
            glbench, "run", "--engine", "glcuda", "--model", MODEL,
            "--prompt", ceiling_prompt, "--tokens", "64", "--cold-iters", "1",
            "--warmup", "2", "--iters", "10", "--temperature", "0", "--seed", "42",
            "--kind", "end_to_end", "--verify-against", "glproc", "--out", out,
        ], cwd=TREE, env=env, check=False)
        save(f"glbench-{index + 1}-{arm}.log", measured)
        if measured.returncode or not out.is_file():
            raise RuntimeError(f"glbench ceiling run failed: {index + 1} {arm}")
        dispatch_log = measured.stdout + "\n" + measured.stderr
        prefill_dispatched = '"path":"wave129-fused-swiglu-production"' in dispatch_log
        decode_dispatched = '"path":"wave132-exact-q8-glue"' in dispatch_log
        contract_enabled = (
            '"q8_nostore":true' in dispatch_log
            and '"ffn_gate_up_stacked":true' in dispatch_log
            and '"defer_ffn_residual":true' in dispatch_log
            and '"n16_fused_swiglu":true' in dispatch_log
        )
        if not contract_enabled or not prefill_dispatched:
            raise RuntimeError(f"retained Wave 131 contract missing: {index + 1} {arm}")
        if (arm == "candidate") != decode_dispatched:
            raise RuntimeError(
                f"decode dispatch mismatch: run={index + 1} arm={arm} dispatched={decode_dispatched}"
            )
        session = json.loads(out.read_text(encoding="utf-8"))
        analysis = session["analysis"]
        validation_text = json.dumps(session.get("validation", {}))
        if not session.get("validation", {}).get("passed") or "50/50 tokens match oracle" not in validation_text:
            raise RuntimeError(f"oracle gate failed: run={index + 1} arm={arm}")
        ceiling_runs.append({
            "order": index + 1,
            "arm": arm,
            "prefill_tps": analysis["prefill_tps"],
            "decode_tps": analysis["decode_tps"],
            "ceiling_efficiency": analysis.get("ceiling_efficiency"),
            "ceiling_basis": analysis.get("ceiling_basis"),
            "bottleneck": analysis.get("bottleneck"),
            "notes": analysis.get("notes", []),
        })

    summary = {
        "wave": 132,
        "candidate": "wave132_exact_decode_q8_glue",
        "gpu": fields,
        "model": model_meta,
        "glbench_ceiling_runs": ceiling_runs,
        "production_ab": {
            "samples_per_arm": len(retained),
            "retained_median_ms": retained_median,
            "candidate_median_ms": candidate_median,
            "retained_tps": 244000.0 / retained_median,
            "candidate_tps": 244000.0 / candidate_median,
            "speedup": retained_median / candidate_median,
            "all_candidate_deltas_positive": all(c < r for c, r in zip(candidate, retained)),
            "invocation_summaries": summaries,
        },
        "candidate_profile": profile,
        "candidate_stages": stages,
        "candidate_ranked_stages": ranked,
        "candidate_stage_sum_ms": stage_sum,
        "candidate_attention_ms": sum(by_name[n]["total_ms"] for n in [
            "qkv", "attn_norm", "attn_kv_write", "attention", "attn_out_quant", "attn_out"
        ]),
        "candidate_ffn_ms": sum(by_name[n]["total_ms"] for n in [
            "ffn_down", "ffn_gate_up", "ffn_residual_norm_quant", "ffn_silu_quant",
            "ffn_residual_add"
        ]),
        "retention_authority": True,
        "target_15000_tps_achieved": 244000.0 / candidate_median >= 15000,
    }
    report = [
        "# Wave 131 fused SwiGLU production and ceiling",
        "",
        f"- Production A/B samples: {len(retained)} per arm, counterbalanced a+b",
        f"- Retained median: {retained_median:.6f} ms = {summary['production_ab']['retained_tps']:.1f} tok/s",
        f"- Candidate median: {candidate_median:.6f} ms = {summary['production_ab']['candidate_tps']:.1f} tok/s",
        f"- Speedup: {summary['production_ab']['speedup']:.4f}x",
        f"- Target 15k reached: {summary['target_15000_tps_achieved']}",
        f"- Decode P50 range: {min(r['decode_tps']['median'] for r in ceiling_runs):.3f}.."
        f"{max(r['decode_tps']['median'] for r in ceiling_runs):.3f} tok/s",
        f"- Candidate profile GPU: {profile['gpu_prefill_ms']:.6f} ms = {profile['gpu_prefill_tps']:.1f} tok/s",
        "",
        "| candidate stage | ms | share of GPU total | calls | bytes read | macs |",
        "|---|---:|---:|---:|---:|---:|",
    ]
    for s in ranked:
        report.append(
            f"| `{s['name']}` | {s['total_ms']:.6f} | {100.0 * s['share_of_gpu_total']:.2f}% | "
            f"{s['calls']} | {s['bytes_read']} | {s['macs']} |"
        )
    (RESULTS / "production-records.json").write_text(json.dumps(records, indent=2), encoding="utf-8")
    (RESULTS / "wave131-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    (RESULTS / "REPORT.md").write_text("\n".join(report), encoding="utf-8")
    print("WAVE131_RESULT", json.dumps(summary, indent=2), flush=True)
    archive()
except Exception:
    (RESULTS / "FAILED.json").write_text(json.dumps({
        "phase": phase, "traceback": traceback.format_exc()}, indent=2), encoding="utf-8")
    archive()
    raise
